# Lecture 4 Tutorial: Working with clinical text


## What we will cover

| Part | Topic | Time |
|------|-------|------|
| 0 | Setup, install spaCy | 2 min |
| 1 | Load & explore the data | 4 min |
| 2 | Data quality  | 7 min |
| 3 | Work with dates| 4 min |
| 4 | Extract structure from free text  | 5 min |
| 5 | NLP with spaCy & negspaCy | 10 min |
| ⭐ Advanced | Entity extraction & evaluation | if time allows |

---
## Part 0: Setup

It may take a minute to download the spaCy language model, best to be done before the tutorial.
While it downloads, read ahead to Part 1.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'spacy', 'negspacy', 'gdown'], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)

import pandas as pd
import re
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import spacy
from negspacy.negation import Negex

nlp = spacy.load('en_core_web_sm')
print('spaCy', spacy.__version__, '— model loaded.')

---
## Part 1: Load & explore the data

Run the code block below to download the dataset into Google Colab.


In [ ]:
import gdown
from pathlib import Path

Path('./data').mkdir(exist_ok=True) # ensure the content folder exists for the dataset to be downloaded into
DATASET_DIR = Path("./data") # path of the extracted dataset.
DATASET_FILE = Path("./data/synthetic_hand_wrist_xray_reports_v2.csv") # name of the extracted dataset.

if not DATASET_FILE.exists(): #checks if the dataset is already downloaded in Colab
    gdown.download(id="1ZcHcVlTA8px3rox7cMRSjoUVV4zzX03I", output=str(DATASET_FILE), quiet=False) #Downloads the dataset from a public Google Drive link
    print(f"Dataset ready at {DATASET_DIR}")
else:
    print(f"Dataset already exists at {DATASET_DIR} - skipping download.")

Our dataset has four columns:

| Column | Description |
|--------|-------------|
| `fake_patient_id` | De-identified patient ID |
| `scan_datetime` | When imaging was performed |
| `report_datetime` | When the radiologist signed off |
| `report` | Free-text radiology report |

Each report is semi-structured: an exam header, then labelled sections
(`Indications:`, `Findings:`, `Impression:` / `Conclusion:`).

First, let's load our downloaded dataset, and find out how many reports exist in it, and how many different patients these reports are for.

In [ ]:
df = pd.read_csv('data/synthetic_hand_wrist_xray_reports_v2.csv')
print(f'{len(df)} rows, {df["fake_patient_id"].nunique()} unique patients')
df.head(3)

Let's pretty-print two reports to see its structure.

In [ ]:
def pretty_print(report):
    formatted = re.sub(
        r'\s*(Indications?|Findings|Impressions?|Conclusions?|Reported By)\s*:',
        r'\n\n\1:',
        report
    )
    print(formatted.strip())

pretty_print(df.loc[0, 'report'])  

In [ ]:
pretty_print(df.loc[1, 'report'])  # Boxer's fracture

### Exercise 1.1. Quick exploration

1. What date range do the scans cover?
2. How many reports contain the word **"fracture"** anywhere?

In [ ]:
df['scan_datetime'] = pd.to_datetime(df['scan_datetime'])
print('Date range:', df['scan_datetime'].min().date(), 'to', df['scan_datetime'].max().date())

fracture_count = df['report'].str.contains('fracture', case=False).sum()
print(f'Reports containing "fracture": {fracture_count} / {len(df)}')

---
## Part 2: Data quality —  Duplicates

Real EHR systems often store the same report more than once.
We identified three patterns during the lecture:

| Type | What it looks like | Why it matters |
|------|--------------------|----------------|
| **Exact duplicate** | Identical text, same patient, same accession | Inflates counts, causes data leakage in train/test splits |
| **Near-duplicate** | Same text with a trailing space, period, or encoding artefact | Won't be caught by exact-match deduplication |
| **Addendum** | A corrected version stored as a new row days later | If you keep the original you may train on the wrong label |

### 2.1. Identify patients with more than one row

In [ ]:
patient_counts = df['fake_patient_id'].value_counts()
print(f'Total rows: {len(df)}')
print(f'Unique patients: {len(patient_counts)}')
print()
print('Patients with more than one row:')
print(patient_counts[patient_counts > 1])

In [ ]:
# Print all rows for each duplicated patient
dup_patients = patient_counts[patient_counts > 1].index.tolist()

for pid in dup_patients:
    print(f'\n=== {pid} ===')
    for _, row in df[df['fake_patient_id'] == pid].iterrows():
        print(f'  report_datetime: {row["report_datetime"]}')
        print(f'  report start: {row["report"][:120]}...')
        print()

### 2.2. Step 1: Handle addendums

Addendums can appear differently depending on your EHR.
In this tutorial, addendum rows start with (or contains) the word **"Addendum"**.

In [ ]:
df['report_datetime'] = pd.to_datetime(df['report_datetime'])

# Flag addendum rows
df['is_addendum'] = df['report'].str.contains(r'\baddendum\b', case=False, na=False)
print('Addendum rows found:')
print(df[df['is_addendum']][['fake_patient_id', 'report_datetime', 'report']].to_string())

In [ ]:
# For any patient who has an addendum: drop all their non-addendum rows
patients_with_addendum = df[df['is_addendum']]['fake_patient_id'].unique()

keep_mask = ~(
    df['fake_patient_id'].isin(patients_with_addendum) & ~df['is_addendum']
)
df = df[keep_mask].drop(columns='is_addendum').reset_index(drop=True)
print(f'After addendum cleanup: {len(df)} rows')

### 2.3. Step 2: Remove exact and near-exact duplicates

After handling addendums, we still have patients with multiple identical (or nearly identical) rows.

**Normalise first**, then deduplicate:
1. Strip leading/trailing whitespace
2. Collapse any run of whitespace to a single space
3. Remove trailing punctuation or spaces 

> **Note:** These normalisation steps are only used to identify duplicates.



In [ ]:
def normalise(text):
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)          # collapse internal whitespace
    text = re.sub(r'[\s.]+$', '', text)        # strip trailing spaces / periods
    return text

df['report_norm'] = df['report'].apply(normalise)

# Flag duplicates — keep the last (latest report_datetime if they differ)
df_sorted = df.sort_values('report_datetime')
dup_mask = df_sorted.duplicated(subset=['fake_patient_id', 'report_norm'], keep='last')

print(f'Duplicate rows identified: {dup_mask.sum()}')
print()
print('Rows that will be dropped:')
print(df_sorted[dup_mask][['fake_patient_id', 'report_datetime', 'report']].to_string())

In [ ]:
# Drop duplicates and tidy up
df = (df_sorted[~dup_mask]
      .drop(columns='report_norm')
      .reset_index(drop=True))

print(f'Clean dataset: {len(df)} rows, {df["fake_patient_id"].nunique()} unique patients')

---
## Part 3: Turnaround Time

**Turnaround time** (scan to signed report) is a key radiology performance metric.
Let's calculate it and look at the distribution.

In [ ]:
df['turnaround_hours'] = (
    df['report_datetime'] - df['scan_datetime']
).dt.total_seconds() / 3600

df[['fake_patient_id', 'scan_datetime', 'report_datetime', 'turnaround_hours']].head(5)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df['turnaround_hours'], bins=15, edgecolor='white', color='steelblue')
ax.set_xlabel('Turnaround time (hours)')
ax.set_ylabel('Number of reports')
ax.set_title('Scan to report turnaround time')
plt.tight_layout()
plt.show()
print(df['turnaround_hours'].describe().round(1))

### Exercise 3.1. Fastest and slowest

Find the patient IDs with the **shortest** and **longest** turnaround times.


In [ ]:
fastest_id  = df.loc[df['turnaround_hours'].idxmin(), 'fake_patient_id']
fastest_hrs = df['turnaround_hours'].min()
slowest_id  = df.loc[df['turnaround_hours'].idxmax(), 'fake_patient_id']
slowest_hrs = df['turnaround_hours'].max()

print(f'Fastest: {fastest_id}  ({fastest_hrs:.1f} h)')
print(f'Slowest: {slowest_id}  ({slowest_hrs:.1f} h)')

---
## Part 4: Extracting structure from text 

Each report has three logical sections:

```
Indications: clinical question, why the scan was requested
Findings: what the radiologist observed
Impression/
Conclusion: summary, diagnosis
```

We can split these with a **regular expression** a pattern-matching mini-language
built into Python's `re` module.

In [ ]:
def extract_sections(report):
    HEADERS = r'(Indications?|Findings|Impressions?|Conclusions?|Reported By)'
    splits = [
        (m.group(1), m.start(), m.end())
        for m in re.finditer(HEADERS + r'\s*:', report, re.IGNORECASE)
    ]
    sections = {}
    sections['header'] = report[:splits[0][1]].strip() if splits else report
    for i, (name, _, end) in enumerate(splits):
        next_start = splits[i + 1][1] if i + 1 < len(splits) else len(report)
        content = report[end:next_start].strip()
        KEY_MAP = {
            'indication': 'indication', 'indications': 'indication',
            'findings': 'findings',
            'impression': 'impression', 'impressions': 'impression',
            'conclusion': 'impression', 'conclusions': 'impression',
            'reported by': 'reported_by',
        }
        sections[KEY_MAP.get(name.lower(), name.lower())] = content
    return sections


for k, v in extract_sections(df.loc[0, 'report']).items():
    print(f'[{k}]\n{v}\n')

In [ ]:
# Apply to the whole dataset
parsed = df['report'].apply(extract_sections).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)
df[['fake_patient_id', 'indication', 'findings', 'impression']].head(3)

### Exercise 4.1

The indication almost always contains *"?fracture"* (the clinical question) which is
**not** a confirmed finding. Compare "fracture" mentions in each section.

In [ ]:
in_indication = df['indication'].str.contains('fracture', case=False, na=False).sum()
in_findings   = df['findings'].str.contains('fracture', case=False, na=False).sum()

print(f'"fracture" in Indication : {in_indication}')
print(f'"fracture" in Findings   : {in_findings}')

---
## Part 5: NLP with spaCy & negspaCy

Regex is fast but fragile. Clinical NLP libraries give us proper language understanding.

**spaCy** runs a full Natural Language Processing (NLP) pipeline on text, including steps such as 
tokeniser, sentence splitter, part of speech tagger.You will learn more about it during Day 4.

**negspaCy** adds negation detection on top, using the dependency parse to tell
whether an entity is grammatically negated.

### 5.1. spaCy basics: tokenisation and sentences

In [ ]:
sample_text = df.loc[0, 'findings']
doc = nlp(sample_text)

print('=== Tokens ===')
for token in doc[:12]:
    print(f'  {token.text!r:20s}  pos={token.pos_:6s}  dep={token.dep_}')

print()
print('=== Sentences ===')
for sent in doc.sents:
    print(' ', sent.text)

### 5.2. negspaCy: detecting negated entities

We label "fracture" as a named entity type `FINDING`, then negspaCy checks
whether each entity is grammatically negated.

In [ ]:
if 'entity_ruler' not in nlp.pipe_names:
    ruler = nlp.add_pipe('entity_ruler')
    ruler.add_patterns([
        {'label': 'FINDING', 'pattern': 'fracture'},
        {'label': 'FINDING', 'pattern': 'Fracture'},
    ])

if 'negex' not in nlp.pipe_names:
    nlp.add_pipe('negex', config={'ent_types': ['FINDING']})

test_sentences = [
    'No acute fracture or dislocation.',
    'Acute nondisplaced fracture of the scaphoid waist.',
    'No acute fracture. Boxer fracture of the fifth metacarpal neck.',
]

for sent in test_sentences:
    doc = nlp(sent)
    for ent in doc.ents:
        print(f'  "{sent}"')
        print(f'  -> entity="{ent.text}"  negated={ent._.negex}')
        print()

### Exercise 5.1. Apply to the whole dataset

Use the negspaCy pipeline to classify every report.
A report is **positive** if it contains at least one *non-negated* `FINDING` entity in the findings.

In [ ]:
def spacy_has_fracture(findings):
    if not isinstance(findings, str):
        return False
    doc = nlp(findings)
    return any(ent.label_ == 'FINDING' and not ent._.negex for ent in doc.ents)

df['spacy_positive'] = df['findings'].apply(spacy_has_fracture)

naive = df['findings'].str.contains('fracture', case=False, na=False)
print(f'Naive (keyword only) : {naive.sum()} positive')
print(f'negspaCy             : {df["spacy_positive"].sum()} positive')
print()
print('Turned negative (were false positives):')
for _, row in df[naive & ~df['spacy_positive']].iterrows():
    print(f"  {row['fake_patient_id']}: {row['findings'][:100]}...")

---
## Part 6: Handling uncertainty

Some reports neither confirm nor deny a fracture:
- *"fracture cannot be entirely excluded"*
- *"partly obscured by positioning"*
- *"slight cortical irregularity ... possibly projectional"*

We flag these as `uncertain` rather than lumping them with negatives.

In [ ]:
UNCERTAINTY_PATTERNS = re.compile(
    r'cannot be (entirely |radiographically )?excluded'
    r'|cannot be (fully )?visualised'
    r'|partly obscured'
    r'|(?:subtle|slight|faint) (?:linear )?(?:lucency|density|irregularity|cortical)'
    r'|not entirely normal',
    re.IGNORECASE
)

def classify_report(row) -> str:
    findings  = row['findings']  if isinstance(row['findings'],  str) else ''
    impression = row['impression'] if isinstance(row['impression'], str) else ''
    combined  = findings + ' ' + impression
    if UNCERTAINTY_PATTERNS.search(combined):
        return 'uncertain'
    if row['spacy_positive']:
        return 'positive'
    return 'negative'

df['classification'] = df.apply(classify_report, axis=1)
df['classification'].value_counts()

In [ ]:
counts = df['classification'].value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(counts.index, counts.values, color=['#e74c3c', '#f39c12', '#2ecc71'])
ax.bar_label(bars)
ax.set_title('Report classification\n(section-aware + negspaCy)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### What we built

```
raw CSV (53 rows, some duplicated)
  - addendum detection     
  - text normalisation        
  - exact deduplication       

clean CSV (50 rows, 1 per patient)
  - section split            
  - spaCy + negspaCy          
  - uncertainty regex         
  - label                     
```

---
## ⭐ Advanced Section

*For those who finish early.*

---


### Challenge A: Entity extraction

Beyond a binary label we often want structured facts:
- **Laterality**: left/right/bilateral
- **Bone / site**: distal radius, scaphoid, metacarpal, phalanx, …
- **Displacement**: displaced/undisplaced/comminuted

In [ ]:
LATERALITY   = re.compile(r'\b(left|right|bilateral|lt|rt)\b', re.IGNORECASE)
BONE         = re.compile(
    r'\b(distal radius|scaphoid|metacarpal|phalanx|phalangeal|'
    r'radial styloid|ulnar styloid|triquetral|lunate)\b', re.IGNORECASE
)
DISPLACEMENT = re.compile(
    r'\b((?:minimally |mildly |markedly )?displaced|undisplaced|nondisplaced|'
    r'comminuted|intra-articular|extra-articular)\b', re.IGNORECASE
)

def extract_entities(row):
    text = str(row.get('header', '')) + ' ' + str(row.get('findings', ''))
    return {
        'laterality'  : sorted(set(m.group().lower() for m in LATERALITY.finditer(text))),
        'bones'       : sorted(set(m.group().lower() for m in BONE.finditer(text))),
        'displacement': sorted(set(m.group().lower() for m in DISPLACEMENT.finditer(text))),
    }

for _, row in df[df['classification'] == 'positive'].head(6).iterrows():
    print(f"{row['fake_patient_id']}: {extract_entities(row)}")

---
### Challenge B: Temporality

*"Previous fracture, now healed"* is not the current injury.

Modify `spacy_has_fracture()` to also skip fracture mentions preceded by
temporal qualifiers like *previous, prior, old, healed, chronic, remote*.

In [ ]:
HISTORICAL = re.compile(
    r'\b(previous|prior|old|healed|chronic|remote|known|longstanding)\b',
    re.IGNORECASE
)

def spacy_has_fracture_v2(findings: str) -> bool:
    if not isinstance(findings, str):
        return False
    doc = nlp(findings)
    for ent in doc.ents:
        if ent.label_ != 'FINDING' or ent._.negex:
            continue
        window = findings[max(0, ent.start_char - 60): ent.start_char]
        if HISTORICAL.search(window):
            continue
        return True
    return False


test_cases = [
    ('Previous distal radius fracture, well healed. No new acute fracture.', False),
    ('Acute nondisplaced fracture of the scaphoid waist.',                   True),
    ('No acute fracture. Known old fracture of the radial styloid.',         False),
    ('No acute fracture seen. Boxer fracture of the fifth metacarpal neck.', True),
]

for text, expected in test_cases:
    result = spacy_has_fracture_v2(text)
    status = 'OK  ' if result == expected else 'FAIL'
    print(f'{status}  expected={expected}  got={result}   "{text[:70]}"')

---
### Challenge C: scispaCy 

`en_core_web_sm` is a general English model. For clinical text, **scispaCy** provides
models trained on biomedical literature with better handling of medical abbreviations.

```bash
pip install scispacy
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
```

Try replacing `spacy.load('en_core_web_sm')` with `spacy.load('en_core_sci_sm')` and
compare sentence splitting.